# DemandLens: exploratory data analysis

This notebook examines the UCI Online Retail II dataset (Chen, 2012) before modelling: its structure, its data-quality problems, and the demand characteristics that shape the choice of methods. Every step uses the same code as the DemandLens application (`analytics/`), so what is reported here is what the system does.

In [ ]:
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analytics.config import AnalysisConfig
from analytics.ingest import read_online_retail, clean_online_retail
from analytics.preprocess import build_panel

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .3})
GREEN, INK, AMBER, BRICK = "#1E6B5C", "#1B2B3A", "#B7791F", "#A63D2F"

# The UCI Online Retail II file (CSV or the original .xlsx). Set DEMANDLENS_DATA to use another file,
# for example the synthetic file from scripts/generate_synthetic.py.
candidates = [os.environ.get("DEMANDLENS_DATA"), ROOT / "data/online_retail_II.csv", ROOT / "data/online_retail_II.xlsx"]
DATA = next(Path(p) for p in candidates if p and Path(p).exists())
print("Using", DATA)

## 1. The raw data

In [ ]:
raw = read_online_retail(DATA)
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"])
print(raw.shape)
display(raw.head())
summary = pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "missing %": (raw.isna().mean() * 100).round(2),
                        "unique": raw.nunique()})
display(summary)
print("Date range:", raw["InvoiceDate"].min(), "to", raw["InvoiceDate"].max())

In [ ]:
display(raw.describe()[["Quantity", "Price"]].T)
top_countries = raw["Country"].value_counts(normalize=True).head(8).mul(100).round(1)
display(top_countries.rename("% of rows"))

## 2. Data-quality problems

Cancellations (invoice numbers starting with C), negative quantities, zero prices, non-product codes such as postage, and exact duplicates all need a rule before the data can be treated as demand.

In [ ]:
is_cancel = raw["Invoice"].astype(str).str.startswith("C")
non_product = ~raw["StockCode"].astype(str).str.match(r"^\d{5}[A-Za-z]{0,3}$")
quality = pd.Series({
    "cancellation rows": is_cancel.sum(),
    "negative quantity, not a cancellation": ((raw["Quantity"] < 0) & ~is_cancel).sum(),
    "zero or negative price": (raw["Price"] <= 0).sum(),
    "non-product codes": non_product.sum(),
    "exact duplicate rows": raw.duplicated().sum(),
    "missing customer id": raw["Customer ID"].isna().sum(),
})
display(quality.to_frame("rows").assign(pct=lambda d: (d["rows"] / len(raw) * 100).round(2)))
display(raw.loc[non_product, "StockCode"].value_counts().head(12).rename("non-product codes"))

In [ ]:
# The largest single order lines, and whether they were cancelled soon after.
display(raw.nlargest(6, "Quantity")[["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Customer ID"]])

## 3. Cleaning into the ledger

The rules in `analytics/ingest.py`: drop non-product codes and duplicates; drop zero-price sales; void orders cancelled in full by the same customer within 90 days; treat other cancellations as returns; keep negative non-cancellation lines as stock adjustments (not demand); drop customer identifiers.

In [ ]:
raw = read_online_retail(DATA)
movements, products, report = clean_online_retail(raw)
movements["product_id"] = pd.factorize(movements["sku"])[0] + 1     # stand-in ids (the app uses database ids)
sku_of = movements.drop_duplicates("product_id").set_index("product_id")["sku"]
name_of = products.set_index("sku")["name"]
panel = build_panel(movements)
cfg = AnalysisConfig()
T = len(panel.weeks)
print(f"{len(movements):,} ledger movements, {panel.units.shape[0]:,} products, {T} complete weeks "
      f"({panel.weeks[0].date()} to {panel.last_week.date()})")
display(pd.Series({k: v for k, v in report.items() if isinstance(v, (int, float))}, name="count").to_frame())

## 4. Sales over time

In [ ]:
rev = np.nansum(panel.revenue, axis=0); units = np.nansum(panel.units, axis=0)
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(panel.weeks, units, color=INK); ax[0].set_ylabel("units / week")
ax[1].plot(panel.weeks, rev, color=GREEN); ax[1].set_ylabel("revenue / week")
fig.suptitle("Weekly demand (net of returns)"); plt.tight_layout(); plt.show()

In [ ]:
wk = pd.DataFrame({"week": panel.weeks, "units": units})
wk["year"] = wk["week"].dt.isocalendar().year; wk["woy"] = wk["week"].dt.isocalendar().week
pivot = wk.pivot_table(index="woy", columns="year", values="units")
pivot.plot(title="Weekly units by week of the year: does the seasonal pattern repeat?", colormap="viridis"); plt.xlabel("ISO week"); plt.show()
print("Correlation between years on common weeks:")
display(pivot.corr().round(2))

In [ ]:
sales = movements[movements["type"] == "SALE"].copy()
sales["weekday"] = sales["occurred_at"].dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sales.groupby("weekday")["quantity"].sum().reindex(order).plot.bar(color=GREEN, title="Units sold by weekday"); plt.show()

## 5. Revenue concentration (ABC)

In [ ]:
from analytics.segmentation import profile
prof = profile(panel, T - 1, cfg)
live = prof[prof["revenue_52w"] > 0].sort_values("revenue_52w", ascending=False)
cum = live["revenue_52w"].cumsum() / live["revenue_52w"].sum()
share = np.arange(1, len(live) + 1) / len(live)
plt.plot(share * 100, cum.values * 100, color=GREEN); plt.plot([0, 100], [0, 100], "--", color="grey")
plt.xlabel("% of products (highest revenue first)"); plt.ylabel("% of revenue"); plt.title("Pareto curve, last 52 weeks"); plt.show()
display(live.groupby("abc_class").agg(products=("product_id", "size"), revenue_share=("revenue_share", "sum")).round(3))

## 6. How demand behaves week to week

The share of zero weeks and the Syntetos-Boylan classification decide which forecasting methods are appropriate.

In [ ]:
obs = ~np.isnan(panel.units[:, -52:])
zero_share = ((np.nan_to_num(panel.units[:, -52:]) == 0) & obs).sum(1) / np.maximum(obs.sum(1), 1)
plt.hist(zero_share[obs.sum(1) >= 26], bins=30, color=GREEN)
plt.xlabel("share of weeks with no sales"); plt.ylabel("products"); plt.title("Intermittency, products with 26+ weeks of history"); plt.show()

In [ ]:
p = prof[prof["demand_pattern"] != "insufficient"]
colors = {"smooth": GREEN, "erratic": AMBER, "intermittent": INK, "lumpy": BRICK}
for pat, g in p.groupby("demand_pattern"):
    plt.scatter(g["adi"], g["cv2"], s=6, alpha=.5, color=colors[pat], label=f"{pat} ({len(g)})")
plt.axvline(cfg.adi_cutoff, color="grey", ls="--"); plt.axhline(cfg.cv2_cutoff, color="grey", ls="--")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("ADI (average weeks between sales)"); plt.ylabel("CV² of sale sizes")
plt.title("Syntetos-Boylan demand classes"); plt.legend(); plt.show()
display(prof["demand_pattern"].value_counts().rename("products"))

## 7. Price and quantity

In [ ]:
long = panel.long.dropna(subset=["avg_price"]).copy()
long["rel_price"] = long["avg_price"] / long.groupby("product_id")["avg_price"].transform("median")
long["units_rel"] = long["units"] / long.groupby("product_id")["units"].transform("mean")
bins = pd.cut(long["rel_price"], [0, .7, .85, .95, 1.05, 1.15, 1.3, 5])
long.groupby(bins, observed=True)["units_rel"].median().plot.bar(color=GREEN)
plt.xlabel("price relative to the product's usual price"); plt.ylabel("median units vs product mean")
plt.title("Lower prices go with higher sales (a reason for the price feature)"); plt.show()

## 8. Returns and trends

In [ ]:
by_type = movements.groupby("type")["quantity"].sum()
print(f"Returned units as a share of units sold: {by_type.get('RETURN', 0) / by_type['SALE']:.2%}")
from analytics.trends import classify_trends
tr = classify_trends(panel, T - 1, cfg)
display(tr["trend"].value_counts().rename("products (Mann-Kendall, last 26 weeks, 5%)"))

## 9. Implications for modelling

Use the outputs above to write up the findings. The questions they answer:

* **Seasonality** (section 4): if the pattern repeats across years, week-of-year and last-year's-sales features are justified.
* **Concentration** (section 5): how few products carry most revenue, which is why forecast accuracy on A products matters most.
* **Intermittency** (section 6): a large share of intermittent and lumpy products rules out per-product ARIMA and motivates Croston's method as a baseline and a global model that learns across products.
* **Price** (section 7): whether relative price moves with demand, which supports the price feature.
* **Data quality** (sections 2-3): every rule and its count, for the preprocessing chapter.